# Text Summarization with Transformers

This notebook demonstrates how to fine-tune a pretrained sequence-to-sequence (Seq2Seq) Transformer model for abstractive text summarization. It covers dataset preparation, preprocessing, evaluation with ROUGE, supervised fine-tuning, and custom training using the Accelerate library.

## Learning Objectives

- Understand abstractive text summarization
- Prepare summarization datasets
- Tokenize source documents and summaries
- Evaluate summarization models using ROUGE
- Fine-tune pretrained Seq2Seq models
- Train using both the Trainer API and Accelerate

## Technologies

- Python
- PyTorch
- Hugging Face Transformers
- Hugging Face Datasets
- Hugging Face Evaluate
- Accelerate

## Preparing the Summarization Dataset

The first step is to prepare a dataset containing document-summary pairs. In this section, we load the dataset, inspect its structure, tokenize both the source documents and target summaries, and convert them into numerical representations suitable for sequence-to-sequence training.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [ ]:
from datasets import DatasetDict

small_dataset = DatasetDict()

for split in dataset.keys():
    n = int(0.01 * len(dataset[split]))
    small_dataset[split] = dataset[split].select(range(n))

small_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 2871
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 133
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 114
    })
})

In [ ]:
def show_samples(dataset, num_samples=3, seed=42):
    sample = dataset["train"].shuffle(seed=seed).select(range(num_samples))
    for example in sample:
        print(f"\n'>> highlights: {example['highlights']}'")
        print(f"'>> article: {example['article']}'")


show_samples(small_dataset)


'>> highlights: Superman, Batman, others born during Great Depression, early World War II years .
New exhibit shows "golden age" of superhero characters .
Artist Jerry Robinson, who created The Joker: "We're looking for heroes"'
'>> article: LOS ANGELES, California (CNN) -- America faces an economic calamity. Trouble brews in faraway lands. Superman #14, cover art. Artist: Fred Ray. (c) 1941 DC Comics. All rights reserved. Sound familiar? More than 70 years ago, the very first superheroes debuted in the dire times of the Great Depression and the early years of World War II. Their names became legend -- Superman, Batman (or, as he was then known, the Bat-Man), Wonder Woman, Captain America -- and they're still with us today. A new exhibit at the Skirball Cultural Center in Los Angeles celebrates these icons from the Golden Age of Comic Books. Through a collection of rare original artwork and comics, the exhibit explores how a group of mostly Jewish artists created the costumed heroes w

In [ ]:
small_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 2871
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 133
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 114
    })
})

In [ ]:
small_dataset = small_dataset.filter(lambda x: len(x["highlights"].split()) > 2)
small_dataset

Filter:   0%|          | 0/2871 [00:00<?, ? examples/s]

Filter:   0%|          | 0/133 [00:00<?, ? examples/s]

Filter:   0%|          | 0/114 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 2871
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 133
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 114
    })
})

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
inputs = tokenizer("I loved reading the Hunger Games!")
inputs

{'input_ids': [336, 259, 28387, 11807, 287, 62893, 295, 12507, 309, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer.convert_ids_to_tokens(inputs.input_ids)

['▁I', '▁', 'loved', '▁reading', '▁the', '▁Hung', 'er', '▁Games', '!', '</s>']

In [ ]:
max_input_length = 512
max_target_length = 30


def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["article"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        examples["highlights"], max_length=max_target_length, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_datasets = small_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/2871 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

In [ ]:
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

## Evaluating Summarization Quality

Unlike classification tasks, summarization models are evaluated by comparing generated summaries with reference summaries. In this notebook, we use the ROUGE metric to measure the overlap between generated and reference text and establish a baseline before fine-tuning.

In [ ]:
!pip install rouge_score

In [ ]:
import evaluate

rouge_score = evaluate.load("rouge")

In [ ]:
scores = rouge_score.compute(
    predictions=[generated_summary], references=[reference_summary]
)
scores

{'rouge1': np.float64(0.923076923076923),
 'rouge2': np.float64(0.7272727272727272),
 'rougeL': np.float64(0.923076923076923),
 'rougeLsum': np.float64(0.923076923076923)}

In [ ]:
scores["rouge1"]

np.float64(0.923076923076923)

In [ ]:
!pip install nltk

In [ ]:
import nltk

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from nltk.tokenize import sent_tokenize


def three_sentence_summary(text):
    return "\n".join(sent_tokenize(text)[:3])


print(three_sentence_summary(small_dataset["train"][1]["article"]))

Editor's note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events.
Here, Soledad O'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial.
MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor."


In [ ]:
small_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 2871
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 133
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 114
    })
})

In [ ]:
def evaluate_baseline(dataset, metric):
    summaries = [three_sentence_summary(text) for text in dataset["article"]]
    return metric.compute(predictions=summaries, references=dataset["highlights"])

In [ ]:
import pandas as pd

score = evaluate_baseline(small_dataset["validation"], rouge_score)
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, round(score[rn] * 100, 2)) for rn in rouge_names)
rouge_dict

{'rouge1': np.float64(29.73),
 'rouge2': np.float64(12.34),
 'rougeL': np.float64(20.39),
 'rougeLsum': np.float64(27.09)}

## Fine-Tuning the Summarization Model

We initialize a pretrained sequence-to-sequence Transformer model, prepare dynamic batching for variable-length documents, configure the training process, and fine-tune the model using the Hugging Face `Seq2SeqTrainer`. After training, the model is evaluated and can be shared through the Hugging Face Hub.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

In [ ]:
from transformers import Seq2SeqTrainingArguments

batch_size = 8
num_train_epochs = 8
# Show the training loss with every epoch
logging_steps = len(tokenized_datasets["train"]) // batch_size
model_name = model_checkpoint.split("/")[-1]

args = Seq2SeqTrainingArguments(
    output_dir=f"{model_name}-finetuned-amazon-en-es",
    eval_strategy="epoch",
    learning_rate=5.6e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    logging_steps=logging_steps,
    push_to_hub=True,
)

In [ ]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Decode generated summaries into text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # Decode reference summaries into text
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # ROUGE expects a newline after each sentence
    decoded_preds = ["\n".join(sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in decoded_labels]
    # Compute ROUGE scores
    result = rouge_score.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )
    # Extract the median scores
    result = {key: value* 100 for key, value in result.items()}
    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(
    small_dataset["train"].column_names
)

In [ ]:
features = [tokenized_datasets["train"][i] for i in range(2)]
data_collator(features)

{'input_ids': tensor([[ 24728, 177913,    261,  ...,   5176,    295,      1],
        [ 28544,    277,    263,  ...,    259,   6661,      1]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([[ 22930,  51258,   4527,   7330,   8120, 110375,    265,   1689,    263,
           5883,   1117,    511,    259, 124624,    527,    790,   8276,    263,
            812,  15914,    259,    260,  19267,  38519,    259,   6661,    790,
           1070,    375,      1],
        [ 82085,    484,    259,   8174,    281,  63038,    281,  29804,    418,
           6956,    285,    351,    287,    313,   1565,    318,  29280,  17070,
            311,  35831,   1017,  51257,    764,   1398,    823,    259,   6661,
           2250,    418,      1]]), 'decoder_input_ids': tensor([[     0,  22930,  51258,   4527,   7330,   8120, 110375,    265,   1689,
            263,   5883,   1117,    511,    259, 124624,    527,    790,   8276,
            263,   

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_2921/543751272.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,7.329600,3.344287,17.157000,5.025200,13.724100,15.209600
2,3.796300,3.122700,19.356500,6.432300,16.266700,17.629300
3,3.466000,3.042940,20.604200,7.490400,18.010300,18.839300
4,3.305600,2.996216,20.064100,7.524200,17.429300,18.251000
5,3.192900,3.006364,19.907500,6.747700,17.427100,18.142800
6,3.121700,2.983355,20.206000,7.360800,18.017800,18.870700
7,3.086500,2.969972,20.588800,7.713900,18.216500,19.269700
8,3.056600,2.965595,20.201100,7.412200,17.856400,18.876600


TrainOutput(global_step=2872, training_loss=3.7923393780806602, metrics={'train_runtime': 2014.263, 'train_samples_per_second': 11.403, 'train_steps_per_second': 1.426, 'total_flos': 1.214432290013184e+16, 'train_loss': 3.7923393780806602, 'epoch': 8.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 2.96559476852417,
 'eval_rouge1': 20.2011,
 'eval_rouge2': 7.4122,
 'eval_rougeL': 17.8564,
 'eval_rougeLsum': 18.8766,
 'eval_runtime': 14.2867,
 'eval_samples_per_second': 9.309,
 'eval_steps_per_second': 1.19,
 'epoch': 8.0}

In [ ]:
trainer.push_to_hub(tags="Summarization")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...744874.5c6ffcc3d7ee.516.0: 100%|##########| 5.30kB / 5.30kB            

  ...n-en-es/training_args.bin: 100%|##########| 5.97kB / 5.97kB            

  ...amazon-en-es/spiece.model: 100%|##########| 4.31MB / 4.31MB            

  ...n-en-es/model.safetensors:   2%|1         | 24.0MB / 1.20GB            

  ...azon-en-es/tokenizer.json: 100%|##########| 16.4MB / 16.4MB            

  ...47888.5c6ffcc3d7ee.2921.1: 100%|##########|   562B /   562B            

  ...45180.5c6ffcc3d7ee.2921.0: 100%|##########| 11.1kB / 11.1kB            

CommitInfo(commit_url='https://huggingface.co/Miladsaeedi70/mt5-small-finetuned-amazon-en-es/commit/19c475eee0dd0d007aab32f865ae1f1baadb5c6b', commit_message='End of training', commit_description='', oid='19c475eee0dd0d007aab32f865ae1f1baadb5c6b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Miladsaeedi70/mt5-small-finetuned-amazon-en-es', endpoint='https://huggingface.co', repo_type='model', repo_id='Miladsaeedi70/mt5-small-finetuned-amazon-en-es'), pr_revision=None, pr_num=None)

## Custom Training with Accelerate

For greater flexibility, Hugging Face provides the `Accelerate` library, which simplifies custom training loops while supporting efficient execution on CPUs, GPUs, and distributed environments. In this section, we manually configure the training pipeline, optimizer, scheduler, and evaluation workflow.

In [ ]:
tokenized_datasets.set_format("torch")

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

In [ ]:
from torch.utils.data import DataLoader

batch_size = 8
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=batch_size,
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=batch_size
)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [ ]:
from transformers import get_scheduler

num_train_epochs = 10
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    # ROUGE expects a newline after each sentence
    preds = ["\n".join(nltk.sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(nltk.sent_tokenize(label)) for label in labels]

    return preds, labels

In [ ]:
from huggingface_hub import get_full_repo_name

model_name = "test-bert-finetuned-squad-accelerate"
repo_name = get_full_repo_name(model_name)
repo_name

'lewtun/mt5-finetuned-amazon-en-es-accelerate'

In [ ]:
from huggingface_hub import Repository

output_dir = "results-mt5-finetuned-squad-accelerate"
repo = Repository(output_dir, clone_from=repo_name)

In [ ]:
from tqdm.auto import tqdm
import torch
import numpy as np

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for step, batch in enumerate(train_dataloader):
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )

            generated_tokens = accelerator.pad_across_processes(
                generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
            )
            labels = batch["labels"]

            # If we did not pad to max length, we need to pad the labels too
            labels = accelerator.pad_across_processes(
                batch["labels"], dim=1, pad_index=tokenizer.pad_token_id
            )

            generated_tokens = accelerator.gather(generated_tokens).cpu().numpy()
            labels = accelerator.gather(labels).cpu().numpy()

            # Replace -100 in the labels as we can't decode them
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            if isinstance(generated_tokens, tuple):
                generated_tokens = generated_tokens[0]
            decoded_preds = tokenizer.batch_decode(
                generated_tokens, skip_special_tokens=True
            )
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

            decoded_preds, decoded_labels = postprocess_text(
                decoded_preds, decoded_labels
            )

            rouge_score.add_batch(predictions=decoded_preds, references=decoded_labels)

    # Compute metrics
    result = rouge_score.compute()
    # Extract the median ROUGE scores
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    result = {k: round(v, 4) for k, v in result.items()}
    print(f"Epoch {epoch}:", result)

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)
        repo.push_to_hub(
            commit_message=f"Training in progress epoch {epoch}", blocking=False
        )

Epoch 0: {'rouge1': 5.6351, 'rouge2': 1.1625, 'rougeL': 5.4866, 'rougeLsum': 5.5005}
Epoch 1: {'rouge1': 9.8646, 'rouge2': 3.4106, 'rougeL': 9.9439, 'rougeLsum': 9.9306}
Epoch 2: {'rouge1': 11.0872, 'rouge2': 3.3273, 'rougeL': 11.0508, 'rougeLsum': 10.9468}
Epoch 3: {'rouge1': 11.8587, 'rouge2': 4.8167, 'rougeL': 11.7986, 'rougeLsum': 11.7518}
Epoch 4: {'rouge1': 12.9842, 'rouge2': 5.5887, 'rougeL': 12.7546, 'rougeLsum': 12.7029}
Epoch 5: {'rouge1': 13.4628, 'rouge2': 6.4598, 'rougeL': 13.312, 'rougeLsum': 13.2913}
Epoch 6: {'rouge1': 12.9131, 'rouge2': 5.8914, 'rougeL': 12.6896, 'rougeLsum': 12.5701}
Epoch 7: {'rouge1': 13.3079, 'rouge2': 6.2994, 'rougeL': 13.1536, 'rougeLsum': 13.1194}
Epoch 8: {'rouge1': 13.96, 'rouge2': 6.5998, 'rougeL': 13.9123, 'rougeLsum': 13.7744}
Epoch 9: {'rouge1': 14.1192, 'rouge2': 7.0059, 'rougeL': 14.1172, 'rougeLsum': 13.9509}

In [ ]:
from transformers import pipeline

hub_model_id = "huggingface-course/mt5-small-finetuned-amazon-en-es"
summarizer = pipeline("summarization", model=hub_model_id)

## Generating Summaries

After fine-tuning, we use the summarization pipeline to generate summaries for unseen documents and compare the generated outputs with the reference summaries to assess model performance qualitatively.

In [ ]:
def print_summary(idx):
    review = books_dataset["test"][idx]["review_body"]
    title = books_dataset["test"][idx]["review_title"]
    summary = summarizer(books_dataset["test"][idx]["review_body"])[0]["summary_text"]
    print(f"'>>> Review: {review}'")
    print(f"\n'>>> Title: {title}'")
    print(f"\n'>>> Summary: {summary}'")

In [ ]:
print_summary(100)

'>>> Review: Nothing special at all about this product... the book is too small and stiff and hard to write in. The huge sticker on the back doesn’t come off and looks super tacky. I would not purchase this again. I could have just bought a journal from the dollar store and it would be basically the same thing. It’s also really expensive for what it is.'

'>>> Title: Not impressed at all... buy something else'

'>>> Summary: Nothing special at all about this product'

In [ ]:
print_summary(0)

'>>> Review: Es una trilogia que se hace muy facil de leer. Me ha gustado, no me esperaba el final para nada'

'>>> Title: Buena literatura para adolescentes'

'>>> Summary: Muy facil de leer'

---

# Key Takeaways

In this notebook, I learned how to:

- Prepare document-summary datasets for sequence-to-sequence learning.
- Tokenize source documents and target summaries.
- Evaluate summarization models using the ROUGE metric.
- Fine-tune pretrained Transformer models for abstractive summarization.
- Implement both high-level (`Seq2SeqTrainer`) and custom (`Accelerate`) training workflows.
- Generate summaries for previously unseen documents.